In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load Data

df = pd.read_csv('diabetic_data.csv')
ids_map = pd.read_csv('IDS_mapping.csv')

print("Original Shape:", df.shape)
print(df.head(3))

Original Shape: (101766, 50)
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No      No                   No   
1                 3  ...          No      Up                   No   
2                 2  ...          No      No                   No   

   glipizide-metformin  glimepiride-pioglitazone  metformin-rosiglitazone  \
0                   No                 

In [ ]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

print("\nMissing values per column (before cleaning):")
print(df.isnull().sum()[df.isnull().sum() > 0])


Missing values per column (before cleaning):
race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64


In [ ]:
# Drop Columns
# 'weight' is missing in ~97% of rows — not usable
# 'encounter_id' and 'patient_nbr' are IDs, not features
df.drop(columns=['weight', 'encounter_id', 'patient_nbr'], inplace=True)

print("Dropped: weight, encounter_id, patient_nbr")

Dropped: weight, encounter_id, patient_nbr


In [ ]:
# Fill Missing Values
df['race'].fillna('Unknown', inplace=True)
df['medical_specialty'].fillna('Unknown', inplace=True)
df['payer_code'].fillna('Unknown', inplace=True)
df['diag_1'].fillna('0', inplace=True)
df['diag_2'].fillna('0', inplace=True)
df['diag_3'].fillna('0', inplace=True)

print("Filled missing: race, medical_specialty, payer_code, diag_1/2/3")

Filled missing: race, medical_specialty, payer_code, diag_1/2/3


/tmp/ipykernel_7682/3213847701.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['race'].fillna('Unknown', inplace=True)
/tmp/ipykernel_7682/3213847701.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

In [ ]:
# Remove Invalid Rows
# Remove unknown/invalid gender entries
df = df[df['gender'] != 'Unknown/Invalid']

# Remove patients who died (discharge_disposition_id == 11)
# Readmission is meaningless for deceased patients
df = df[df['discharge_disposition_id'] != 11]

print("Removed: invalid gender rows, deceased patients (discharge=11)")

Removed: invalid gender rows, deceased patients (discharge=11)


In [ ]:
# Encode Target Variable
# Binary: readmitted within 30 days = 1, otherwise = 0
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

print("\nReadmission distribution:")
print(df['readmitted'].value_counts())
print("\nBinary target distribution:")
print(df['readmitted_binary'].value_counts())


Readmission distribution:
readmitted
NO     53219
>30    35545
<30    11357
Name: count, dtype: int64

Binary target distribution:
readmitted_binary
0    88764
1    11357
Name: count, dtype: int64


In [ ]:
# Map Age to Numeric
age_map = {
    '[0-10)': 5,  '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age_numeric'] = df['age'].map(age_map)

print("Added age_numeric column")

Added age_numeric column


In [ ]:
# Fix Column Types
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
            'num_medications', 'number_outpatient', 'number_emergency',
            'number_inpatient', 'number_diagnoses']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

print("Converted numeric columns")

Converted numeric columns


In [ ]:
# Final Check & Save
print("\nCleaned Shape:", df.shape)
print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.to_csv('diabetic_cleaned.csv', index=False)
print("\n Saved cleaned dataset!")


Cleaned Shape: (100121, 49)
Remaining missing values:
max_glu_serum    94896
A1Cresult        83244
dtype: int64

 Saved cleaned dataset!
